# 05 — AlphaFold/OpenFold3: ERAP2 Structure Prediction

**Goal:** Predict 3D structures of wild-type and mutant ERAP2.

Options:
1. **AlphaFold DB lookup** — ERAP2 (Q6P179) is already in the database
2. **OpenFold3** — predict mutant structure from sequence
3. **ESMFold** — fast structure prediction from sequence alone (no MSA)
4. **Boltz-2** — structure + binding affinity prediction

**Validation:** Compare prediction against PDB experimental structure (if available).
Target RMSD < 2 Angstroms.

**Run on:** Google Colab A100

In [ ]:
# Install dependencies
!pip install -q transformers torch accelerate biopython py3Dmol

In [ ]:
import requests
import json

# Step 1: Check if ERAP2 is in AlphaFold DB (it should be)
AF_DB_URL = "https://alphafold.ebi.ac.uk/api/prediction/Q6P179"
resp = requests.get(AF_DB_URL)
if resp.ok:
    af_data = resp.json()
    print(f"AlphaFold DB entry found for ERAP2 (Q6P179)")
    if isinstance(af_data, list) and af_data:
        entry = af_data[0]
        print(f"  Model URL: {entry.get('pdbUrl', 'N/A')}")
        print(f"  Confidence: {entry.get('modelCreatedDate', 'N/A')}")
        pdb_url = entry.get('pdbUrl', '')
        cif_url = entry.get('cifUrl', '')
    else:
        print(f"  Data: {af_data}")
else:
    print(f"Not in AlphaFold DB: {resp.status_code}")

In [ ]:
# Step 2: Download the AlphaFold structure
if pdb_url:
    pdb_resp = requests.get(pdb_url)
    if pdb_resp.ok:
        with open("erap2_wt_alphafold.pdb", "w") as f:
            f.write(pdb_resp.text)
        print("Downloaded wild-type ERAP2 structure")
        print(f"Size: {len(pdb_resp.text)} characters")

In [ ]:
# Step 3: Also check PDB for experimental ERAP2 structures
PDB_SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
pdb_query = {
    "query": {
        "type": "group",
        "logical_operator": "and",
        "nodes": [
            {
                "type": "terminal",
                "service": "text",
                "parameters": {
                    "attribute": "rcsb_polymer_entity.pdbx_description",
                    "operator": "contains_words",
                    "value": "endoplasmic reticulum aminopeptidase 2"
                }
            }
        ]
    },
    "return_type": "entry"
}

resp = requests.post(PDB_SEARCH_URL, json=pdb_query)
if resp.ok:
    pdb_results = resp.json()
    hits = pdb_results.get("result_set", [])
    print(f"Found {len(hits)} experimental PDB structures for ERAP2")
    for hit in hits[:10]:
        pdb_id = hit.get("identifier", "")
        print(f"  {pdb_id} — https://www.rcsb.org/structure/{pdb_id}")
else:
    print(f"PDB search failed: {resp.status_code}")

In [ ]:
# Step 4: ESMFold for fast mutant structure prediction
# (Alternative to full OpenFold3 — faster, less accurate but good for comparison)
from transformers import AutoTokenizer, EsmForProteinFolding
from transformers.models.esm.openfold_utils.protein import to_pdb, Protein

# Load ESMFold
tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
model = model.cuda()
model.eval()
print("ESMFold loaded")

In [ ]:
# Predict structures for WT and mutant
# TODO: Paste sequences from script 02 output
ERAP2_WT_SEQ = ""  # Wild-type ERAP2 sequence
ERAP2_MT_SEQ = ""  # Mutant ERAP2 sequence (with rs2549794 amino acid change)

def predict_structure(sequence, name):
    """Predict structure with ESMFold and save PDB."""
    inputs = tokenizer([sequence], return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.no_grad():
        output = model(**inputs)
    
    # Convert to PDB
    pdb_string = to_pdb(Protein(
        aatype=output["aatype"][0],
        atom_positions=output["positions"][-1][0],
        atom_mask=output["atom37_atom_exists"][0],
        residue_index=torch.arange(len(sequence)),
        b_factors=output["plddt"][0],
    ))
    
    with open(f"erap2_{name}_esmfold.pdb", "w") as f:
        f.write(pdb_string)
    
    # Mean pLDDT (confidence)
    mean_plddt = output["plddt"][0].mean().item()
    print(f"{name}: mean pLDDT = {mean_plddt:.1f}")
    return pdb_string, mean_plddt

if ERAP2_WT_SEQ:
    wt_pdb, wt_conf = predict_structure(ERAP2_WT_SEQ, "wt")
    if ERAP2_MT_SEQ:
        mt_pdb, mt_conf = predict_structure(ERAP2_MT_SEQ, "mutant")
else:
    print("Upload ERAP2 sequences from local pipeline output first.")

In [ ]:
# Step 5: Visualize structures (3D in notebook)
import py3Dmol

def show_structure(pdb_string, label=""):
    view = py3Dmol.view(width=800, height=500)
    view.addModel(pdb_string, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    if label:
        view.addLabel(label, {"fontSize": 14, "position": {"x": 0, "y": 0, "z": 0}})
    return view.show()

# Visualize after prediction
# show_structure(wt_pdb, "ERAP2 Wild-Type")